<a href="https://colab.research.google.com/github/Bhagyashreegarje07/AI-Fashion-Designer/blob/main/AI_FASHION_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q gradio fal-client google-generativeai requests

In [ ]:
import gradio as gr
import fal_client
import google.generativeai as genai
import requests
import os

# --- API SETUP ---
# Get your keys from: fal.ai, aistudio.google.com, and serpapi.com
os.environ["FAL_KEY"] = "YOUR_FAL_API_KEY"
genai.configure(api_key="YOUR_GEMINI_API_KEY")
SERP_API_KEY = "YOUR_SERPAPI_KEY"

def fashion_bot(user_prompt, style, fabric, budget):
    # 1. Generate the Fashion Design
    refined_prompt = f"Professional fashion photography, {style} style, {fabric} material, {user_prompt}, high resolution, studio lighting"

    try:
        # Using Flux model for high-end fashion realism
        handler = fal_client.submit(
            "fal-ai/flux/schnell",
            arguments={"prompt": refined_prompt}
        )
        result = handler.get()
        image_url = result['images'][0]['url']
    except Exception as e:
        return None, f"Error generating image: {e}", []

    # 2. Analyze Design with Gemini to find search keywords
    model = genai.GenerativeModel('gemini-1.5-flash')
    vision_prompt = f"Identify the specific clothing item in this prompt: {user_prompt}. Provide a 3-word shopping search term (e.g., 'blue silk blazer')."
    analysis = model.generate_content(vision_prompt)
    search_query = analysis.text.strip()

    # 3. Search for affordable matches via SerpApi (Google Shopping)
    shopping_results = []
    search_url = f"https://serpapi.com/search.json?engine=google_shopping&q={search_query}&price_max={budget}&api_key={SERP_API_KEY}"

    try:
        response = requests.get(search_url).json()
        if "shopping_results" in response:
            for item in response["shopping_results"][:3]:
                product_info = f"🛒 {item['title'][:40]}...\n💰 Price: {item['price']}\n🔗 {item['link']}"
                shopping_results.append(product_info)
    except:
        shopping_results = ["Could not fetch shopping results at this time."]

    shopping_text = "\n\n".join(shopping_results) if shopping_results else "No matches found."

    return image_url, f"**Search Query Used:** {search_query}\n\n**Affordable Matches:**\n{shopping_text}"

# --- GRADIO UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 👗 AI Fashion Internship Project")
    gr.Markdown("Describe a garment, and I'll design it + find affordable matches online.")

    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(label="Describe your design", placeholder="e.g. A futuristic silver puffer jacket")
            style_input = gr.Dropdown(["Streetwear", "Minimalist", "Cyberpunk", "Vintage", "Boho"], label="Style", value="Streetwear")
            fabric_input = gr.Dropdown(["Cotton", "Silk", "Denim", "Leather", "Linen"], label="Fabric", value="Cotton")
            budget_input = gr.Slider(10, 500, value=100, label="Max Budget ($)")
            btn = gr.Button("Generate Design")

        with gr.Column():
            output_img = gr.Image(label="AI Generated Design")
            output_text = gr.Markdown(label="Shopping Suggestions")

    btn.click(fn=fashion_bot,
              inputs=[prompt_input, style_input, fabric_input, budget_input],
              outputs=[output_img, output_text])

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipython-input-211/2155581547.py:52: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://535dd421d8a9bd7577.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
